# HashiCorp Vault: Static vs Dynamic Secrets for Cloud IAM Credential Management

> First-day notebook exploring how Vault hands out cloud credentials. I wrote this as I worked through the examples.

## Purpose

Cloud IAM credentials are the most painful secret to manage: long-lived access keys in env vars, rotation that nobody does, and an audit trail that stops at the CI/CD job. Vault's cloud secrets engines (AWS, Azure, GCP) offer a different path — short-lived, Vault-issued principals. This notebook compares the static and dynamic approaches side by side.

## Static vs Dynamic: The Core Difference

| | Static (KV) | Dynamic (Cloud engine) |
|---|---|---|
| Credential lifetime | Until someone rotates it | TTL-based, e.g. 1h |
| Rotation | Manual or scripted | Automatic on lease expiry |
| Blast radius | The key you committed | A single session |
| Audit | Read/write events | Issue + revoke events |
| Overhead | Low to start | Engine + role config |

Static secrets are fine for third-party API keys you can't rotate yourself. For cloud IAM, dynamic is the safer default.

## Setting Up the AWS Engine

The AWS secrets engine needs a privileged IAM user or role that Vault uses to call the AWS API. That principal never appears in the notebook or in CI logs.

## A tiny example

I ran this against a Vault dev server with `VAULT_ADDR=http://127.0.0.1:8200` and `VAULT_TOKEN=root`.

bash
# Enable the engine and configure the root credential
vault secrets enable -path=aws-iam aws
vault write aws-iam/config/root \
  access_key=$AWS_ACCESS_KEY_ID \
  secret_key=$AWS_SECRET_ACCESS_KEY

# Create a role that issues short-lived IAM users
vault write aws-iam/roles/ci-role \
  credential_type=iam_user \
  policy_arns=arn:aws:iam::aws:policy/ReadOnlyAccess \
  default_sts_ttl=1h \
  max_sts_ttl=24h

# Generate credentials and read the TTL
vault read -format=json aws-iam/creds/ci-role | jq '{access_key, lease_id, lease_duration}'

The response shows a 3600-second lease. Revoking the lease removes the IAM user from AWS.

## What I learned

- The dynamic engine needs a privileged principal — it never issues credentials it can't create itself.
- `credential_type=iam_user` creates a real IAM user; `credential_type=assumed_role` returns STS tokens instead.
- Applications must re-read credentials before the TTL expires, or use Vault Agent for sidecar-based renewal.

## What I'll cover next

Next I'll look at the database secrets engine and how it compares to the cloud engine, then at lease renewal patterns in a long-running workload.


In [ ]:
# Compare static vs dynamic secrets for cloud IAM

import json

static = {
    "type": "access_key",
    "lifetime": "until rotated",
    "rotation": "manual"
}

dynamic = {
    "type": "sts_token",
    "lifetime": "1h",
    "rotation": "automatic"
}

print(json.dumps({"static": static, "dynamic": dynamic}, indent=2))

# Run this cell to see the side-by-side comparison.

# The key insight: with dynamic secrets, the credential you hold
# is only valid for its lease duration, so the blast radius of a
# leak is bounded by the TTL rather than the lifetime of the key.
